# Pilot analysis — dependência de contexto em clarificação com LLMs

## 0. Contexto e natureza do piloto

**Este estudo é um piloto metodológico. As estatísticas apresentadas são descritivas e exploratórias e têm como objetivo avaliar a viabilidade do método e identificar padrões preliminares. Não devem ser interpretadas como estimativas generalizáveis do comportamento de LLMs ou do Spec Kit.**

Escopo deste notebook: operacionalizar o método, construir unidades analíticas, métricas descritivas e infraestrutura. Não executar Codex/Spec Kit; não alterar dados experimentais, PRR ou anotações humanas; não inferir mapping semântico pergunta→gap.


## 1. Imports e configuração

In [1]:
import sys
from pathlib import Path
from datetime import datetime, timezone

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent.parent
elif (REPO_ROOT / "analysis" / "config.py").exists():
    pass
elif (REPO_ROOT.parent / "analysis" / "config.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis import config
from analysis.src.loaders import environment_versions, load_analysis_inputs
from analysis.src.validation import AnalysisValidationError, validate_all_inputs
from analysis.src.mappings import build_final_mapping, explode_question_gap_mapping
from analysis.src.matrices import build_gap_run_matrix
from analysis.src.metrics import (
    classify_repetition_stability,
    dataset_description,
    format_rate,
    gap_recall,
    occurrence_rate,
    questions_summary_stats,
    requery_rate,
    viability_metrics,
)
from analysis.src import plots
from analysis.src.statistics import (
    clustered_bootstrap_ci,
    exploratory_inferential_models_disabled,
    fit_mixed_logit_recognized_future,
)
from analysis.src.pipeline import run_pipeline

print("ANALYSIS_SEED =", config.ANALYSIS_SEED)
print("timestamp UTC:", datetime.now(timezone.utc).isoformat())
print("versions:", environment_versions())
print(config.PILOT_DISCLAIMER)


ANALYSIS_SEED = 20260930
timestamp UTC: 2026-10-01T01:45:20.862628+00:00
versions: {'python': '3.13.9', 'pandas': '2.3.3', 'numpy': '2.3.5', 'matplotlib': '3.10.6', 'scipy': '1.16.3', 'statsmodels': '0.14.5'}
Este estudo é um piloto metodológico. As estatísticas apresentadas são descritivas e exploratórias e têm como objetivo avaliar a viabilidade do método e identificar padrões preliminares. Não devem ser interpretadas como estimativas generalizáveis do comportamento de LLMs ou do Spec Kit.


## 2–6. Carregamento, validação, mapping e Gap×Run

In [2]:
# Pipeline completo: valida inputs → mapping final → question-gap-mapping → gap-run-matrix → tabelas/figuras.
# Se validação crítica falhar, PARA com mensagem clara (não corrige dados humanos).

try:
    artifacts = run_pipeline(pilot_asserts=True, write_outputs=True)
except AnalysisValidationError as err:
    print(str(err))
    raise SystemExit("ANÁLISE INTERROMPIDA: corrija/complete a anotação humana antes de continuar.")

print("Input hashes:")
for k, v in artifacts.meta["input_hashes"].items():
    print(f"  {k}: {v[:16]}…")

final_map = artifacts.final_map
qgm = artifacts.question_gap_mapping
matrix = artifacts.gap_run_matrix
print(f"perguntas={len(final_map)} qgm_rows={len(qgm)} gap_run_rows={len(matrix)}")
assert len(matrix) == config.PILOT_EXPECTED_GAP_RUN_ROWS
assert (matrix["recognized"].isin([0, 1])).all()
assert (matrix["question_count_for_gap"] >= matrix["recognized"]).all()
assert (matrix["gap_id"] != "NONE").all()
c0 = matrix[matrix["condition"] == "C0"]
assert (c0["gap_state"] == "OPEN").all(), "C0 deve ter todos os gaps OPEN"


Input hashes:
  questions: 1b8d3517f941c324…
  evaluator_1: f5ef01a1980cb73c…
  evaluator_2: de32046a42af2e35…
  disagreements: ae4fc15f83904cf6…
  blind_id_map: de37f1f0ec9b7b37…
  prr_reference: ff1214a994857cc5…
  prr_gap_states: 692a2217471ba8cd…
  run_summary: 1481ed52e1ccf78c…
  prr_workbook: 43de71862dc3f045…
perguntas=269 qgm_rows=302 gap_run_rows=756


## 7. Descrição do dataset

In [3]:
desc = artifacts.meta["dataset"]
display(artifacts.tables["dataset-summary"])
display(artifacts.tables["questions-by-condition"])
print("Gaps por US:", desc["gaps_by_user_story"])
print("Gaps por categoria:", desc.get("gaps_by_category"))
print("Gaps por importance:", desc.get("gaps_by_importance"))


,metric,value
0,n_user_stories,4.000000
1,n_gaps,42.000000
2,n_runs,72.000000
3,n_questions,269.000000
4,direct_agreement,106.000000
5,adjudicated,163.000000
6,none_count,66.000000
7,none_rate,0.245353
8,multi_gap_count,28.000000
9,gap_run_matrix_rows,756.000000


,condition,n_questions
0,C0,55
1,CD,42
2,CL,48
3,CO,43
4,CS,42
5,CT,39


Gaps por US: {'US02': 11, 'US08': 10, 'US18': 10, 'US25': 11}
Gaps por categoria: {'Decisória': 4, 'Léxica': 7, 'Operacional': 23, 'Sistêmica': 8}
Gaps por importance: {'Complementar': 5, 'Necessária': 37}


## 8. Viabilidade do método de anotação

Divergência entre avaliadores é reportada apenas como **diagnóstico do procedimento de anotação**.
Não se reporta Cohen's kappa como evidência de confiabilidade interavaliadores neste piloto (Avaliador 2 assistivo/ChatGPT).


In [4]:
via = artifacts.meta["viability"]
display(artifacts.tables["viability-metrics"])
print(via["note"])
print(
    f"Acordos diretos={via['direct_agreement']}; "
    f"adjudicados={via['adjudicated']}; "
    f"burden={format_rate(via['adjudication_burden'])}; "
    f"NONE rate={format_rate(via['none_rate'])}; "
    f"multi-gap rate={format_rate(via['multi_gap_rate'])}"
)


,n_questions,direct_agreement,adjudicated,adjudication_burden,direct_agreement_rate,none_count,none_rate,multi_gap_count,multi_gap_rate,prr_mapped_count,prr_mapped_rate,note
0,269,106,163,0.605948,0.394052,66,0.245353,28,0.104089,203,0.754647,Divergência reportada apenas como diagnóstico ...


Divergência reportada apenas como diagnóstico do procedimento de anotação. Não usar Cohen's kappa como evidência de confiabilidade interavaliadores neste piloto (Avaliador 2 assistivo/ChatGPT).
Acordos diretos=106; adjudicados=163; burden=0.6059; NONE rate=0.2454; multi-gap rate=0.1041


## 9. RQ1 — baseline / C0 (somente OPEN)

Gap Recall run-level = recognized OPEN / total OPEN (ocorrências Gap×Run).
Miss Rate = 1 − Gap Recall na mesma unidade (não é descoberta independente).


In [5]:
display(artifacts.tables["rq1-c0-global"])
display(artifacts.tables["gap-recall-c0-by-user-story"])
display(artifacts.tables["gap-recall-c0-by-importance"])
display(artifacts.tables["gap-recall-c0-by-category"])
r = artifacts.tables["rq1-c0-global"].iloc[0]["gap_recall"]
print(f"No piloto, a condição C0 apresentou Gap Recall de {format_rate(r)} (unidade Gap×Run OPEN).")


,open_gaps,recognized_open_gaps,gap_recall,miss_rate
0,126,47,0.373016,0.626984


,user_story_id,open_gaps,recognized_open_gaps,gap_recall,miss_rate
0,US02,33,10,0.303030,0.696970
1,US08,30,6,0.200000,0.800000
2,US18,30,17,0.566667,0.433333
3,US25,33,14,0.424242,0.575758


,importance,open_gaps,recognized_open_gaps,gap_recall,miss_rate
0,Complementar,15,0,0.000000,1.000000
1,Necessária,111,47,0.423423,0.576577


,gap_category,open_gaps,recognized_open_gaps,gap_recall,miss_rate
0,Decisória,12,0,0.000000,1.000000
1,Léxica,21,7,0.333333,0.666667
2,Operacional,69,33,0.478261,0.521739
3,Sistêmica,24,7,0.291667,0.708333


No piloto, a condição C0 apresentou Gap Recall de 0.3730 (unidade Gap×Run OPEN).


## 10. RQ2 — gaps remanescentes OPEN (CL, CO, CD, CS; CT pode ser N/A)

In [6]:
display(artifacts.tables["gap-recall-rq2-by-condition"])
display(artifacts.tables["gap-recall-by-condition"])
# N/A nunca convertido para zero
for _, row in artifacts.tables["gap-recall-by-condition"].iterrows():
    print(f"{row['condition']}: open={row['open_gaps']} recall={format_rate(row['gap_recall'])}")


,condition,open_gaps,recognized_open_gaps,gap_recall,miss_rate
0,CD,114,32,0.280702,0.719298
1,CL,105,36,0.342857,0.657143
2,CO,57,2,0.035088,0.964912
3,CS,102,30,0.294118,0.705882


,condition,open_gaps,recognized_open_gaps,gap_recall,miss_rate
0,C0,126,47,0.373016,0.626984
1,CD,114,32,0.280702,0.719298
2,CL,105,36,0.342857,0.657143
3,CO,57,2,0.035088,0.964912
4,CS,102,30,0.294118,0.705882


C0: open=126 recall=0.3730
CD: open=114 recall=0.2807
CL: open=105 recall=0.3429
CO: open=57 recall=0.0351
CS: open=102 recall=0.2941


## 11. RQ3 — reconsulta de gaps ANSWERED

In [7]:
display(artifacts.tables["requery-rate-global"])
display(artifacts.tables["requery-rate-by-condition"])
display(artifacts.tables["requery-rate-by-user-story"])
display(artifacts.tables["requery-rate-by-category"])


,answered_gaps,requery_count,requery_rate
0,252,79,0.313492


,condition,answered_gaps,requery_count,requery_rate
0,CD,12,0,0.000000
1,CL,21,8,0.380952
2,CO,69,37,0.536232
3,CS,24,0,0.000000
4,CT,126,34,0.269841


,user_story_id,answered_gaps,requery_count,requery_rate
0,US02,66,23,0.348485
1,US08,60,9,0.150000
2,US18,60,16,0.266667
3,US25,66,31,0.469697


,gap_category,answered_gaps,requery_count,requery_rate
0,Decisória,24,0,0.000000
1,Léxica,42,12,0.285714
2,Operacional,138,65,0.471014
3,Sistêmica,48,2,0.041667


## 12–13. Occurrence Rate e estabilidade entre repetições (rótulos DESCRITIVOS)

In [8]:
display(artifacts.tables["gap-occurrence-rate"].head(20))
display(artifacts.tables["repetition-stability"]["stability_label"].value_counts())
print("Rótulos 0/3…3/3 são DESCRITIVOS do piloto, não categorias universais.")


,user_story_id,gap_id,condition,gap_state,occurrences,repetitions,occurrence_rate
0,US02,G01,C0,OPEN,0,3,0.000000
1,US02,G01,CD,OPEN,0,3,0.000000
2,US02,G01,CL,ANSWERED,0,3,0.000000
3,US02,G01,CO,OPEN,0,3,0.000000
4,US02,G01,CS,OPEN,0,3,0.000000
5,US02,G01,CT,ANSWERED,0,3,0.000000
6,US02,G02,C0,OPEN,1,3,0.333333
7,US02,G02,CD,OPEN,1,3,0.333333
8,US02,G02,CL,OPEN,1,3,0.333333
9,US02,G02,CO,ANSWERED,1,3,0.333333


stability_label
nunca reconhecido             142
reconhecimento consistente     45
reconhecimento raro            39
reconhecimento frequente       26
Name: count, dtype: int64

Rótulos 0/3…3/3 são DESCRITIVOS do piloto, não categorias universais.


## 14. NONE / gaps não previstos (somente quantitativo)

In [9]:
display(artifacts.tables["unmatched-summary"])
display(artifacts.tables["none-by-user-story"])
display(artifacts.tables["none-by-condition"])
print("Análise qualitativa de NONE fica para estudo exploratório separado.")


,total_none,none_rate
0,66,0.245353


,user_story_id,none_count,none_rate
0,US02,13,0.178082
1,US08,29,0.517857
2,US18,13,0.216667
3,US25,11,0.137500


,condition,none_count,none_rate
0,C0,14,0.254545
1,CD,11,0.261905
2,CL,10,0.208333
3,CO,8,0.186047
4,CS,14,0.333333
5,CT,9,0.230769


Análise qualitativa de NONE fica para estudo exploratório separado.


## 15. Número bruto de perguntas

**Menor quantidade de perguntas não significa necessariamente melhor clarificação.**
Não usar raw question count como métrica principal de qualidade.


In [10]:
display(artifacts.tables["question-count-stats-per-run"])
print(artifacts.meta["question_count_warning"])


,mean,median,std,min,max,iqr
0,3.736111,4.0,0.992834,1.0,5.0,1.0


Menor quantidade de perguntas não significa necessariamente melhor clarificação.


## Bootstrap clusterizado (exploratório) e modelos inferenciais desabilitados

In [11]:
print(artifacts.meta["clustered_bootstrap_c0_recall"])
print(artifacts.meta["inferential_models"])
print(artifacts.meta["mixed_logit_future"])


{'estimate': 0.373015873015873, 'n_clusters': 4, 'n_boot': 2000, 'seed': 20260930, 'alpha': 0.05, 'ci_low': None, 'ci_high': None, 'status': 'skipped_insufficient_clusters', 'note': 'Bootstrap clusterizado não reportado: n_clusters=4 < min_clusters_for_ci=10. Função pronta para o estudo principal com mais User Stories / gaps.'}
{'section': 'Exploratory inferential models — disabled for pilot', 'disabled': 'Friedman, Wilcoxon, ANOVA, Kruskal-Wallis, and pairwise batteries are NOT executed automatically in this pilot.', 'rationale': 'Pilot is methodological validation with n=4 User Stories; confirmatory inference would be inappropriate.', 'future_options': 'For the main study: pre-register contrasts; use mixed models or cluster-robust inference; control multiplicity; avoid treating Gap×Run rows as independent.'}
{'status': 'experimental/future', 'enabled': False, 'message': 'Mixed-effects logistic model infrastructure reserved for the main study. Not fitted in the pilot.', 'outcome': 're

## 16–18. Síntese, limitações e implicações para o experimento principal

Interpretação apenas descritiva (sem linguagem causal/confirmatória).


In [12]:
for note in artifacts.descriptive_notes:
    print("-", note)
print()
print("Limitações: n=4 US; Avaliador 2 assistivo; CIs clusterizados tipicamente inadequados no piloto;")
print("sem testes confirmatórios automáticos; OccurrenceRate/GLMM prontos para escala futura.")
print("Implicação: completar anotação humana → reexecutar ./scripts/bin/run-analysis.sh no estudo principal com mais US.")
print("Figuras:", len(artifacts.figure_paths))
print("Tabelas:", sorted(artifacts.tables))


- Este estudo é um piloto metodológico. As estatísticas apresentadas são descritivas e exploratórias e têm como objetivo avaliar a viabilidade do método e identificar padrões preliminares. Não devem ser interpretadas como estimativas generalizáveis do comportamento de LLMs ou do Spec Kit.
- No piloto, a condição C0 apresentou Gap Recall de 0.3730 (unidade: ocorrências Gap×Run OPEN).
- Em CT, a taxa de reconsulta (ANSWERED) foi 0.2698.

Limitações: n=4 US; Avaliador 2 assistivo; CIs clusterizados tipicamente inadequados no piloto;
sem testes confirmatórios automáticos; OccurrenceRate/GLMM prontos para escala futura.
Implicação: completar anotação humana → reexecutar ./scripts/bin/run-analysis.sh no estudo principal com mais US.
Figuras: 20
Tabelas: ['dataset-summary', 'gap-occurrence-rate', 'gap-recall-by-condition', 'gap-recall-by-user-story', 'gap-recall-c0-by-category', 'gap-recall-c0-by-importance', 'gap-recall-c0-by-user-story', 'gap-recall-open-by-category', 'gap-recall-open-by-co